# 라이브러리 및 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [3]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_friendrequest`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

df.head()

KeyboardInterrupt: 

- 신청시 created_at이 생성되고 동일한 값으로 updated_at이 생성되며, 상태(status)값이 업데이트 되는 순간 updated_at이 수정되는 것으로 보임.

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17147175 entries, 0 to 17147174
Data columns (total 6 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               int64         
 1   status           str           
 2   created_at       datetime64[us]
 3   updated_at       datetime64[us]
 4   receive_user_id  int64         
 5   send_user_id     int64         
dtypes: datetime64[us](2), int64(3), str(1)
memory usage: 801.3 MB


- 총 17,147,175 행 확인.
- 날짜형태도 결측없이 존재하는 것으로 특이값은 없다고 판단.

## 결측 체크
- 결측 없음.

In [ ]:
df.isna().sum()

id                 0
status             0
created_at         0
updated_at         0
receive_user_id    0
send_user_id       0
dtype: int64

## 중복 체크
- 중복 없음.
- 유저아이디 기준 친구 요청 및 요청 수신 유저 모두 요청 건수에 따라서 중복되는 결과 확인. 따로 처리하지 않음.

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df['send_user_id'].nunique()

649072

## 컬럼별 특이값 확인

In [ ]:
df.columns

Index(['id', 'status', 'created_at', 'updated_at', 'receive_user_id',
       'send_user_id'],
      dtype='str')

In [ ]:
df['status'].value_counts()

status
A    12878407
P     3938608
R      330160
Name: count, dtype: int64

- A (수락), P (대기), R (거절) 범주 이상없음.

In [ ]:
print(df['created_at'].min())
print(df['created_at'].max())

2023-04-17 18:29:11
2024-05-09 09:21:47


In [ ]:
print(f"보낸 유저 아이디 중 가장 낮은 유저 아이디 : {df['send_user_id'].min()}")
print(f"보낸 유저 아이디 중 가장 큰 유저 아이디 : {df['send_user_id'].max()}")

print(f"수신 유저 아이디 중 가장 낮은 유저 아이디 : {df['receive_user_id'].min()}")
print(f"수신 유저 아이디 중 가장 큰 유저 아이디 : {df['receive_user_id'].max()}")

보낸 유저 아이디 중 가장 낮은 유저 아이디 : 831962
보낸 유저 아이디 중 가장 큰 유저 아이디 : 1583732
수신 유저 아이디 중 가장 낮은 유저 아이디 : 831962
수신 유저 아이디 중 가장 큰 유저 아이디 : 1583731


- 숫자형 아이디의 범위가 모두 유저 테이블에 부합한다고 판단. 특이값 없음.

## 정리
- 모든 테이블의 날짜 데이터의 기준은 확인필요.
- 다른 전처리 작업은 없음. 그대로 유지하는 것을 결론으로 수정 및 처리 코딩없이 유지함.

# 친구 관계 형성 테이블 제작
* 친구 요청 기록과 요청 상태 등을 파악하여, 친구 관계 형성을 트래킹 할 수 있는 데이터 제작
* 'A' 상태값을 가진 행이 관계가 형성되었다고 판단.
* 관계 형성 시기를 나타내야 하므로, updated_at를 활용

In [6]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT send_user_id, receive_user_id, updated_at
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_friendrequest`
    WHERE status = 'A'
    ORDER BY updated_at ASC
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

df.head()

,send_user_id,receive_user_id,updated_at
0,837521,832340,2023-04-18 19:28:41+00:00
1,837532,837530,2023-04-19 06:06:31+00:00
2,837543,837531,2023-04-19 06:08:19+00:00
3,837538,837531,2023-04-19 06:08:20+00:00
4,837537,837531,2023-04-19 06:08:21+00:00


In [28]:
df.shape

(12878407, 3)

## 셀프 친구 관계 확인

In [44]:
self_friendships = df[
    df['send_user_id'].eq(
        df['receive_user_id']
    )
].copy()

print(
    f'자기 자신에게 친구 신청한 기록 수: '
    f'{len(self_friendships):,}건'
)

display(self_friendships.head(20))

자기 자신에게 친구 신청한 기록 수: 0건


,send_user_id,receive_user_id,updated_at


## 중복 친구 관계 확인

In [45]:
friendship_check = df.copy()

friendship_check['user_id_1'] = friendship_check[
    ['send_user_id', 'receive_user_id']
].min(axis=1)

friendship_check['user_id_2'] = friendship_check[
    ['send_user_id', 'receive_user_id']
].max(axis=1)

In [46]:
friendship_duplicate_summary = (
    friendship_check
    .groupby(
        ['user_id_1', 'user_id_2'],
        as_index=False,
    )
    .agg(
        수락기록수=('updated_at', 'size'),
        최초수락시각=('updated_at', 'min'),
        최종수락시각=('updated_at', 'max'),
    )
)

duplicate_friendships = (
    friendship_duplicate_summary[
        friendship_duplicate_summary['수락기록수'].gt(1)
    ]
    .sort_values(
        '수락기록수',
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    f'수락 기록이 2건 이상인 친구 관계: '
    f'{len(duplicate_friendships):,}개'
)

display(duplicate_friendships.head(20))

수락 기록이 2건 이상인 친구 관계: 2,960개


,user_id_1,user_id_2,수락기록수,최초수락시각,최종수락시각
0,1264506,1314988,3,2023-05-18 14:08:37+00:00,2023-05-19 15:50:05+00:00
1,1356713,1357864,3,2023-05-18 14:06:26+00:00,2023-05-19 02:38:02+00:00
2,838452,1138849,2,2023-05-18 21:49:54+00:00,2023-05-19 09:18:25+00:00
3,1231352,1340467,2,2023-05-20 15:09:48+00:00,2023-05-22 12:57:08+00:00
4,1230851,1270112,2,2023-05-19 11:19:40+00:00,2023-05-19 15:39:29+00:00
5,1230882,1311674,2,2023-05-18 14:41:50+00:00,2023-09-18 06:32:33+00:00
6,1230896,1239858,2,2023-05-22 13:15:14+00:00,2023-06-03 03:30:19+00:00
7,1230941,1306131,2,2023-05-18 12:54:10+00:00,2023-05-19 13:16:52+00:00
8,1230955,1579196,2,2023-07-26 01:09:52+00:00,2023-07-26 01:09:55+00:00
9,1231088,1231865,2,2023-05-18 23:08:14+00:00,2023-05-19 01:36:57+00:00


In [47]:
print(
    f'중복으로 판단되는 추가 기록 수: '
    f'{(duplicate_friendships["수락기록수"] - 1).sum():,}건'
)

중복으로 판단되는 추가 기록 수: 2,962건


In [48]:
first_accepted_friendships = (
    friendship_check
    .sort_values(
        'updated_at',
        kind='stable',
    )
    .drop_duplicates(
        subset=['user_id_1', 'user_id_2'],
        keep='first',
    )
    .reset_index(drop=True)
)

In [49]:
print(f'중복 처리 전 기록 수: {len(df):,}건')
print(
    f'중복 처리 후 친구 관계 수: '
    f'{len(first_accepted_friendships):,}건'
)

print(
    '중복 처리 후 남은 중복 관계 수:',
    first_accepted_friendships[
        ['user_id_1', 'user_id_2']
    ].duplicated().sum(),
)

중복 처리 전 기록 수: 12,878,407건
중복 처리 후 친구 관계 수: 12,875,445건
중복 처리 후 남은 중복 관계 수: 0


In [50]:
first_accepted_friendships

,send_user_id,receive_user_id,updated_at,user_id_1,user_id_2
0,837521,832340,2023-04-18 19:28:41+00:00,832340,837521
1,837532,837530,2023-04-19 06:06:31+00:00,837530,837532
2,837543,837531,2023-04-19 06:08:19+00:00,837531,837543
3,837538,837531,2023-04-19 06:08:20+00:00,837531,837538
4,837537,837531,2023-04-19 06:08:21+00:00,837531,837537
...,...,...,...,...,...
12875440,1110948,1197989,2024-05-09 05:21:40+00:00,1110948,1197989
12875441,1054750,1197989,2024-05-09 05:21:40+00:00,1054750,1197989
12875442,1583732,1583731,2024-05-09 07:25:52+00:00,1583731,1583732
12875443,1583732,1583730,2024-05-09 07:32:54+00:00,1583730,1583732


## 친구 관계 형성 확인

In [51]:
# user_id_1 관점
user_1_events = (
    first_accepted_friendships[
        ['user_id_1', 'user_id_2', 'updated_at']
    ]
    .rename(
        columns={
            'user_id_1': 'user_id',
            'user_id_2': 'friend_user_id',
            'updated_at': 'friendship_at',
        }
    )
)

# user_id_2 관점
user_2_events = (
    first_accepted_friendships[
        ['user_id_1', 'user_id_2', 'updated_at']
    ]
    .rename(
        columns={
            'user_id_2': 'user_id',
            'user_id_1': 'friend_user_id',
            'updated_at': 'friendship_at',
        }
    )
)

In [52]:
friend_count_log = pd.concat(
    [
        user_1_events,
        user_2_events,
    ],
    ignore_index=True,
)

friend_count_log = (
    friend_count_log
    .sort_values(
        [
            'user_id',
            'friendship_at',
            'friend_user_id',
        ]
    )
    .reset_index(drop=True)
)

In [53]:
friend_count_log['friend_cnt_before'] = (
    friend_count_log
    .groupby('user_id')
    .cumcount()
)

friend_count_log['friend_cnt_after'] = (
    friend_count_log['friend_cnt_before'] + 1
)

In [54]:
display(friend_count_log.head(20))

,user_id,friend_user_id,friendship_at,friend_cnt_before,friend_cnt_after
0,831962,1446852,2023-09-11 15:38:43+00:00,0,1
1,832151,841037,2023-04-22 06:02:44+00:00,0,1
2,832151,838785,2023-04-22 06:02:46+00:00,1,2
3,832151,837950,2023-04-22 06:02:48+00:00,2,3
4,832151,838541,2023-04-22 06:02:51+00:00,3,4
5,832151,837521,2023-04-22 06:02:53+00:00,4,5
6,832151,836498,2023-04-22 13:39:22+00:00,5,6
7,832151,840046,2023-04-23 08:57:27+00:00,6,7
8,832151,862823,2023-05-16 04:44:32+00:00,7,8
9,832340,837521,2023-04-18 19:28:41+00:00,0,1


In [55]:
print(
    f'최초 친구 관계 수: '
    f'{len(first_accepted_friendships):,}개'
)

print(
    f'예상 로그 수: '
    f'{len(first_accepted_friendships) * 2:,}개'
)

print(
    f'실제 로그 수: '
    f'{len(friend_count_log):,}개'
)

최초 친구 관계 수: 12,875,445개
예상 로그 수: 25,750,890개
실제 로그 수: 25,750,890개


In [ ]:
friend_count_log[friend_count_log['user_id'] == 1446852]

,user_id,friend_user_id,friendship_at,friend_cnt_before,friend_cnt_after
22169057,1446852,838541,2023-06-20 12:17:20+00:00,0,1
22169058,1446852,849763,2023-06-20 12:17:33+00:00,1,2
22169059,1446852,1577131,2023-06-25 05:33:23+00:00,2,3
22169060,1446852,1578013,2023-07-11 04:50:46+00:00,3,4
22169061,1446852,836498,2023-07-11 07:54:35+00:00,4,5
22169062,1446852,1578661,2023-07-15 04:25:33+00:00,5,6
22169063,1446852,1579656,2023-08-08 12:28:04+00:00,6,7
22169064,1446852,1437875,2023-08-19 05:01:44+00:00,7,8
22169065,1446852,1577954,2023-08-19 15:46:10+00:00,8,9
22169066,1446852,1579542,2023-08-21 12:35:15+00:00,9,10


# 최종 테이블

In [58]:
friend_count_log[['user_id', 'friend_user_id', 'friend_cnt_after', 'friendship_at']]

,user_id,friend_user_id,friend_cnt_after,friendship_at
0,831962,1446852,1,2023-09-11 15:38:43+00:00
1,832151,841037,1,2023-04-22 06:02:44+00:00
2,832151,838785,2,2023-04-22 06:02:46+00:00
3,832151,837950,3,2023-04-22 06:02:48+00:00
4,832151,838541,4,2023-04-22 06:02:51+00:00
...,...,...,...,...
25750885,1583730,1583732,1,2024-05-09 07:32:54+00:00
25750886,1583731,1583732,1,2024-05-09 07:25:52+00:00
25750887,1583731,1583673,2,2024-05-09 07:33:06+00:00
25750888,1583732,1583731,1,2024-05-09 07:25:52+00:00


# 테이블 생성

In [60]:
table_id = (
    f'{PROJECT_ID}.{DATA_SET}.user_friend_count_history'
)

job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            'user_id',
            'INTEGER',
            mode='REQUIRED',
        ),
        bigquery.SchemaField(
            'friend_user_id',
            'INTEGER',
            mode='REQUIRED',
        ),
        bigquery.SchemaField(
            'friendship_at',
            'TIMESTAMP',
            mode='REQUIRED',
        ),
        bigquery.SchemaField(
            'friend_cnt_before',
            'INTEGER',
            mode='REQUIRED',
        ),
        bigquery.SchemaField(
            'friend_cnt_after',
            'INTEGER',
            mode='REQUIRED',
        ),
    ],
    write_disposition='WRITE_EMPTY',
)

In [61]:
load_job = client.load_table_from_dataframe(
    friend_count_log[
        [
            'user_id',
            'friend_user_id',
            'friendship_at',
            'friend_cnt_before',
            'friend_cnt_after',
        ]
    ],
    table_id,
    job_config=job_config,
)

load_job.result()

destination_table = client.get_table(table_id)

print(
    f'{table_id}에 '
    f'{destination_table.num_rows:,}개 행 업로드 완료'
)

sns-analysis-prj.sns_analysis.user_friend_count_history에 25,750,890개 행 업로드 완료
